In [7]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = os.getcwd() if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Repo already exists — pulling latest changes
Already up to date.


In [8]:
%%capture
if not IS_LOCAL:
    !pip install optuna

import optuna

In [9]:
import importlib
import scipy.sparse as sps
import pandas as pd
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import hyperparameter_tuning

Running on kaggle — storage at: /kaggle/working


In [10]:
# Load datasets
URM_train = sps.load_npz(paths.URM_TRAIN)
URM_validation = sps.load_npz(paths.URM_VALIDATION)

In [11]:
def evaluate_recommender(recommender, at):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

# Train a KNN for each similarity

["cosine", "pearson", "jaccard", "tversky"]

In [32]:
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender

KKN_MODEL_DIR = os.path.join(paths.MODEL_DIR, "ItemKNN")
os.makedirs(KKN_MODEL_DIR,  exist_ok=True)

In [13]:
def perform_optimization(similarity, n_trials):
    # Define objective function for hyperparameter tuning
    STUDY_NAME = ItemKNNCFRecommender.RECOMMENDER_NAME + "_" + similarity
    
    def objective_function(optuna_trial: optuna.trial.Trial) -> float:
        recommender_instance = ItemKNNCFRecommender(URM_train)
        recommender_instance.fit(
            similarity=similarity,
            topK=optuna_trial.suggest_int("topK", 10, 1500),
            shrink=optuna_trial.suggest_int("shrink", 0, 2000),
            normalize=optuna_trial.suggest_categorical("normalize", [True, False]),
            feature_weighting=optuna_trial.suggest_categorical("feature_weighting", ["BM25", "TF-IDF", "none"])
        )
    
        return evaluate_recommender(recommender_instance, at=20)

    # Perform hyperparameter tuning
    save_results, optuna_study = hyperparameter_tuning(
        objective_function,
        study_name=STUDY_NAME,
        n_trials=n_trials
    )

    return save_results, optuna_study

In [20]:
def perfome_fine_tuning(similarity, n_trials, best_params):
    # Define objective function for hyperparameter tuning
    STUDY_NAME = ItemKNNCFRecommender.RECOMMENDER_NAME + "_" + similarity

    normalize = best_params['normalize']
    feature_weighting=best_params['feature_weighting']
    topK_min = max(0, best_params['topK'] - 10)
    topK_max = best_params['topK'] + 10
    shrink_min = max(0, best_params['shrink'] - 20)
    shrink_max = best_params['shrink'] + 20
    
    def objective_function(optuna_trial: optuna.trial.Trial) -> float:
        recommender_instance = ItemKNNCFRecommender(URM_train)
        recommender_instance.fit(
            similarity=similarity,
            topK=optuna_trial.suggest_int("topK", topK_min, topK_max),
            shrink=optuna_trial.suggest_int("shrink", shrink_min, shrink_max),
            normalize=normalize,
            feature_weighting=feature_weighting
        )
    
        return evaluate_recommender(recommender_instance, at=20)

    # Perform hyperparameter tuning
    save_results, optuna_study = hyperparameter_tuning(
        objective_function,
        study_name=STUDY_NAME,
        n_trials=n_trials
    )

    return save_results, optuna_study

## Cosine

In [14]:
SIMILARITY = "cosine"

In [16]:
fd_results, optuna_study = perform_optimization(SIMILARITY, 100)

  0%|          | 0/100 [00:00<?, ?it/s]

Similarity column 6969 (100.0%), 1877.22 column/sec. Elapsed time 3.71 sec
[I 2025-11-08 14:55:08,720] Trial 0 finished with value: 0.1656958970961976 and parameters: {'topK': 854, 'shrink': 1561, 'normalize': False, 'feature_weighting': 'BM25'}. Best is trial 0 with value: 0.1656958970961976.
Similarity column 6969 (100.0%), 1863.37 column/sec. Elapsed time 3.74 sec
[I 2025-11-08 14:55:50,374] Trial 1 finished with value: 0.1660072805503071 and parameters: {'topK': 1306, 'shrink': 1357, 'normalize': False, 'feature_weighting': 'BM25'}. Best is trial 1 with value: 0.1660072805503071.
Similarity column 6969 (100.0%), 1883.14 column/sec. Elapsed time 3.70 sec
[I 2025-11-08 14:56:36,634] Trial 2 finished with value: 0.13574923285365256 and parameters: {'topK': 1111, 'shrink': 164, 'normalize': False, 'feature_weighting': 'TF-IDF'}. Best is trial 1 with value: 0.1660072805503071.
Similarity column 6969 (100.0%), 1856.30 column/sec. Elapsed time 3.75 sec
[I 2025-11-08 14:57:21,816] Trial 3 

In [17]:
optuna.visualization.plot_optimization_history(optuna_study)

In [18]:
optuna.visualization.plot_param_importances(optuna_study)

In [19]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

In [28]:
fd_results, optuna_study = perfome_fine_tuning(SIMILARITY, 20, optuna_study.best_params)

  0%|          | 0/20 [00:00<?, ?it/s]

Similarity column 6969 (100.0%), 1996.95 column/sec. Elapsed time 3.49 sec
[I 2025-11-08 15:55:44,210] Trial 100 finished with value: 0.2177828619060572 and parameters: {'topK': 80, 'shrink': 435}. Best is trial 100 with value: 0.2177828619060572.
Similarity column 6969 (100.0%), 1998.90 column/sec. Elapsed time 3.49 sec
[I 2025-11-08 15:56:05,892] Trial 101 finished with value: 0.2177851504950154 and parameters: {'topK': 80, 'shrink': 433}. Best is trial 101 with value: 0.2177851504950154.
Similarity column 6969 (100.0%), 1962.96 column/sec. Elapsed time 3.55 sec
[I 2025-11-08 15:56:27,717] Trial 102 finished with value: 0.21778418236345984 and parameters: {'topK': 80, 'shrink': 434}. Best is trial 101 with value: 0.2177851504950154.
Similarity column 6969 (100.0%), 1978.13 column/sec. Elapsed time 3.52 sec
[I 2025-11-08 15:56:49,508] Trial 103 finished with value: 0.21778418236345984 and parameters: {'topK': 80, 'shrink': 434}. Best is trial 101 with value: 0.2177851504950154.
Simila

In [34]:
# {'topK': 79, 'shrink': 431, 'normalize': True, 'feature_weighting': 'TF-IDF'}

recommender = ItemKNNCFRecommender(URM_train)
recommender.fit(
    similarity=SIMILARITY,
    topK=79,
    shrink=431,
    normalize=True,
    feature_weighting='TF-IDF'
)

recommender.save_model(KKN_MODEL_DIR, file_name=SIMILARITY)

Similarity column 6969 (100.0%), 1982.62 column/sec. Elapsed time 3.52 sec
ItemKNNCFRecommender: Saving model in file '/kaggle/working/models/ItemKNNcosine'
ItemKNNCFRecommender: Saving complete


### Best Mode
- Best Value: 0.21802981660336454
- Best Params: {'topK': 79, 'shrink': 431, 'normalize': True, 'feature_weighting': 'TF-IDF'}

## Tversky

In [35]:
SIMILARITY = "tversky"

In [36]:
fd_results, optuna_study = perform_optimization(SIMILARITY, 100)

  0%|          | 0/100 [00:00<?, ?it/s]

Similarity column 6969 (100.0%), 1763.15 column/sec. Elapsed time 3.95 sec
[I 2025-11-08 16:10:39,805] Trial 0 finished with value: 0.19723662765730968 and parameters: {'topK': 1204, 'shrink': 853, 'normalize': False, 'feature_weighting': 'BM25'}. Best is trial 0 with value: 0.19723662765730968.
Similarity column 6969 (100.0%), 1924.12 column/sec. Elapsed time 3.62 sec
[I 2025-11-08 16:11:04,357] Trial 1 finished with value: 0.1983665540187874 and parameters: {'topK': 222, 'shrink': 509, 'normalize': True, 'feature_weighting': 'TF-IDF'}. Best is trial 1 with value: 0.1983665540187874.
Similarity column 6969 (100.0%), 1865.08 column/sec. Elapsed time 3.74 sec
[I 2025-11-08 16:11:46,483] Trial 2 finished with value: 0.17078942676654177 and parameters: {'topK': 1090, 'shrink': 1920, 'normalize': False, 'feature_weighting': 'TF-IDF'}. Best is trial 1 with value: 0.1983665540187874.
Similarity column 6969 (100.0%), 1872.31 column/sec. Elapsed time 3.72 sec
[I 2025-11-08 16:12:28,934] Trial 

In [37]:
optuna.visualization.plot_optimization_history(optuna_study)

In [38]:
optuna.visualization.plot_param_importances(optuna_study)

In [39]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

In [41]:
STUDY_NAME = ItemKNNCFRecommender.RECOMMENDER_NAME + "_tuning_" + SIMILARITY

def tversky_tuning_function(optuna_trial: optuna.trial.Trial) -> float:
    recommender_instance = ItemKNNCFRecommender(URM_train)
    recommender_instance.fit(
        similarity=SIMILARITY,
        topK=optuna_trial.suggest_int("topK", 1, 100),
        shrink=optuna_trial.suggest_int("shrink", 20, 100),
        tversky_alpha=optuna_trial.suggest_float("tversky_alpha", 0., 1.),
        tversky_beta=optuna_trial.suggest_float("tversky_beta", 0., 1.),
        normalize=True,
        feature_weighting="TF-IDF"
    )

    return evaluate_recommender(recommender_instance, at=20)

# Perform hyperparameter tuning
save_results, optuna_study = hyperparameter_tuning(
    tversky_tuning_function,
    study_name=STUDY_NAME,
    n_trials=40
)

  0%|          | 0/40 [00:00<?, ?it/s]

Similarity column 6969 (100.0%), 1953.16 column/sec. Elapsed time 3.57 sec
[I 2025-11-08 16:56:55,350] Trial 0 finished with value: 0.1739150533662377 and parameters: {'topK': 93, 'shrink': 46, 'tversky_alpha': 0.1295257866848959, 'tversky_beta': 0.0502658268867433}. Best is trial 0 with value: 0.1739150533662377.
Similarity column 6969 (100.0%), 1930.44 column/sec. Elapsed time 3.61 sec
[I 2025-11-08 16:57:15,216] Trial 1 finished with value: 0.1981408859026984 and parameters: {'topK': 99, 'shrink': 68, 'tversky_alpha': 0.18297804203898926, 'tversky_beta': 0.34763277347522337}. Best is trial 1 with value: 0.1981408859026984.
Similarity column 6969 (100.0%), 1978.59 column/sec. Elapsed time 3.52 sec
[I 2025-11-08 16:57:30,847] Trial 2 finished with value: 0.2310809994712704 and parameters: {'topK': 3, 'shrink': 92, 'tversky_alpha': 0.6334409953176263, 'tversky_beta': 0.982027590999618}. Best is trial 2 with value: 0.2310809994712704.
Similarity column 6969 (100.0%), 1924.31 column/sec.

In [42]:
# Perform hyperparameter tuning
save_results, optuna_study = hyperparameter_tuning(
    tversky_tuning_function,
    study_name=STUDY_NAME,
    n_trials=20
)

  0%|          | 0/20 [00:00<?, ?it/s]

Similarity column 6969 (100.0%), 1958.05 column/sec. Elapsed time 3.56 sec
[I 2025-11-08 17:09:08,240] Trial 40 finished with value: 0.24218963854705647 and parameters: {'topK': 7, 'shrink': 94, 'tversky_alpha': 0.3096587258384417, 'tversky_beta': 0.9652449324621438}. Best is trial 37 with value: 0.24397671252974537.
Similarity column 6969 (100.0%), 1923.14 column/sec. Elapsed time 3.62 sec
[I 2025-11-08 17:09:24,340] Trial 41 finished with value: 0.24444102846833002 and parameters: {'topK': 7, 'shrink': 94, 'tversky_alpha': 0.2759601476054635, 'tversky_beta': 0.9953239633524094}. Best is trial 41 with value: 0.24444102846833002.
Similarity column 6969 (100.0%), 1953.57 column/sec. Elapsed time 3.57 sec
[I 2025-11-08 17:09:40,396] Trial 42 finished with value: 0.2434758455315593 and parameters: {'topK': 7, 'shrink': 94, 'tversky_alpha': 0.29626008575617063, 'tversky_beta': 0.994787859757088}. Best is trial 41 with value: 0.24444102846833002.
Similarity column 6969 (100.0%), 1936.27 col

In [43]:
optuna.visualization.plot_optimization_history(optuna_study)

In [44]:
optuna.visualization.plot_param_importances(optuna_study)

In [45]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

In [47]:
STUDY_NAME = ItemKNNCFRecommender.RECOMMENDER_NAME + "_alpha_beta_1_" + SIMILARITY

def tversky_tuning_alpha_beta(optuna_trial: optuna.trial.Trial) -> float:
    recommender_instance = ItemKNNCFRecommender(URM_train)
    recommender_instance.fit(
        similarity=SIMILARITY,
        topK=optuna_trial.suggest_int("topK", 5, 15),
        shrink=optuna_trial.suggest_int("shrink", 90, 130),
        tversky_alpha=optuna_trial.suggest_float("tversky_alpha", 0., 0.5),
        tversky_beta=optuna_trial.suggest_float("tversky_beta", 0.5, 1.),
        normalize=True,
        feature_weighting="TF-IDF"
    )

    return evaluate_recommender(recommender_instance, at=20)

# Perform hyperparameter tuning
save_results, optuna_study = hyperparameter_tuning(
    tversky_tuning_alpha_beta,
    study_name=STUDY_NAME,
    n_trials=40
)

  0%|          | 0/40 [00:00<?, ?it/s]

Similarity column 6969 (100.0%), 1968.12 column/sec. Elapsed time 3.54 sec
[I 2025-11-08 17:19:28,771] Trial 0 finished with value: 0.2436341597623999 and parameters: {'topK': 6, 'shrink': 125, 'tversky_alpha': 0.18081273139009196, 'tversky_beta': 0.7474665020989189}. Best is trial 0 with value: 0.2436341597623999.
Similarity column 6969 (100.0%), 1952.27 column/sec. Elapsed time 3.57 sec
[I 2025-11-08 17:19:45,864] Trial 1 finished with value: 0.2115409488311645 and parameters: {'topK': 15, 'shrink': 126, 'tversky_alpha': 0.41720585004718524, 'tversky_beta': 0.5359929078203116}. Best is trial 0 with value: 0.2436341597623999.
Similarity column 6969 (100.0%), 1961.54 column/sec. Elapsed time 3.55 sec
[I 2025-11-08 17:20:02,285] Trial 2 finished with value: 0.2391813675972719 and parameters: {'topK': 14, 'shrink': 97, 'tversky_alpha': 0.1846944201225848, 'tversky_beta': 0.8870555932561994}. Best is trial 0 with value: 0.2436341597623999.
Similarity column 6969 (100.0%), 1943.45 column/s

In [48]:
optuna.visualization.plot_optimization_history(optuna_study)

In [49]:
optuna.visualization.plot_param_importances(optuna_study)

In [50]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

In [51]:
# Perform hyperparameter tuning
save_results, optuna_study = hyperparameter_tuning(
    tversky_tuning_alpha_beta,
    study_name=STUDY_NAME,
    n_trials=20
)

  0%|          | 0/20 [00:00<?, ?it/s]

Similarity column 6969 (100.0%), 1933.67 column/sec. Elapsed time 3.60 sec
[I 2025-11-08 17:35:43,522] Trial 40 finished with value: 0.2493087824947337 and parameters: {'topK': 7, 'shrink': 116, 'tversky_alpha': 0.06306808540772597, 'tversky_beta': 0.8690366902842898}. Best is trial 40 with value: 0.2493087824947337.
Similarity column 6969 (100.0%), 1933.52 column/sec. Elapsed time 3.60 sec
[I 2025-11-08 17:35:59,576] Trial 41 finished with value: 0.24878635058515444 and parameters: {'topK': 7, 'shrink': 116, 'tversky_alpha': 0.06085909990151361, 'tversky_beta': 0.8680991054633762}. Best is trial 40 with value: 0.2493087824947337.
Similarity column 6969 (100.0%), 1910.64 column/sec. Elapsed time 3.65 sec
[I 2025-11-08 17:36:15,713] Trial 42 finished with value: 0.24921801119292644 and parameters: {'topK': 7, 'shrink': 120, 'tversky_alpha': 0.06132883646651224, 'tversky_beta': 0.8707858213127302}. Best is trial 40 with value: 0.2493087824947337.
Similarity column 6969 (100.0%), 1937.76 

In [52]:
# Perform hyperparameter tuning
save_results, optuna_study = hyperparameter_tuning(
    tversky_tuning_alpha_beta,
    study_name=STUDY_NAME,
    n_trials=20
)

  0%|          | 0/20 [00:00<?, ?it/s]

Similarity column 6969 (100.0%), 1920.84 column/sec. Elapsed time 3.63 sec
[I 2025-11-08 17:50:11,965] Trial 60 finished with value: 0.2387247155627792 and parameters: {'topK': 8, 'shrink': 115, 'tversky_alpha': 0.31323541476531697, 'tversky_beta': 0.9212358705610202}. Best is trial 54 with value: 0.2496318552097081.
Similarity column 6969 (100.0%), 1923.57 column/sec. Elapsed time 3.62 sec
[I 2025-11-08 17:50:28,048] Trial 61 finished with value: 0.24915834629366673 and parameters: {'topK': 7, 'shrink': 120, 'tversky_alpha': 0.07073023370808093, 'tversky_beta': 0.8795474435946442}. Best is trial 54 with value: 0.2496318552097081.
Similarity column 6969 (100.0%), 1912.87 column/sec. Elapsed time 3.64 sec
[I 2025-11-08 17:50:44,176] Trial 62 finished with value: 0.24741179744403502 and parameters: {'topK': 7, 'shrink': 122, 'tversky_alpha': 0.03504701573752905, 'tversky_beta': 0.8163737837408859}. Best is trial 54 with value: 0.2496318552097081.
Similarity column 6969 (100.0%), 1914.70 

In [53]:
optuna.visualization.plot_optimization_history(optuna_study)

In [54]:
optuna.visualization.plot_param_importances(optuna_study)

In [55]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

### **Best Model**
- Best Value: 0.2505261665362465
- Best Params: {'topK': 6, 'shrink': 106, 'tversky_alpha': 0.10252983277640502, 'tversky_beta': 0.9684276681445425}